# Homework Option B: Conditional Constellation Generator

Time: up to 60 minutes

This notebook builds a 2D point diffusion model conditioned on shape class, using classifier-free guidance. It combines the forward process from Part 1, the reverse process from Part 2, and the CFG formula from Part 3, all on tiny 2D point clouds. Runs on CPU in under a minute.
Look for `# TODO` comments to complete it.
See `Homework_OptionB_ConditionalConstellation.md` for full instructions.

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print('Ready.')

---

## Step 1: A Multi-Shape Dataset

Three shape classes: circle, spiral, and cross. Already working, just run it.

In [ ]:
def make_circle(n, noise=0.05):
    angle = np.random.uniform(0, 2*np.pi, n)
    x = np.cos(angle) + np.random.randn(n) * noise
    y = np.sin(angle) + np.random.randn(n) * noise
    return np.stack([x, y], axis=1)

def make_spiral(n, noise=0.03):
    t = np.random.uniform(0, 4*np.pi, n)
    r = t / (4*np.pi)
    x = r * np.cos(t) + np.random.randn(n) * noise
    y = r * np.sin(t) + np.random.randn(n) * noise
    return np.stack([x, y], axis=1)

def make_cross(n, noise=0.04):
    half = n // 2
    x1 = np.random.uniform(-1, 1, half)
    y1 = np.random.randn(half) * noise
    y2 = np.random.uniform(-1, 1, n - half)
    x2 = np.random.randn(n - half) * noise
    return np.stack([np.concatenate([x1, x2]), np.concatenate([y1, y2])], axis=1)

SHAPE_NAMES = ['circle', 'spiral', 'cross']
N_CLASSES = len(SHAPE_NAMES)
N_PER_CLASS = 700

generators = [make_circle, make_spiral, make_cross]
all_points, all_labels = [], []
for cls_idx, gen in enumerate(generators):
    pts = gen(N_PER_CLASS)
    all_points.append(pts)
    all_labels.append(np.full(N_PER_CLASS, cls_idx))

X = np.concatenate(all_points, axis=0)
X = (X - X.mean(0)) / X.std(0)
y = np.concatenate(all_labels, axis=0)

data   = torch.tensor(X, dtype=torch.float32)
labels = torch.tensor(y, dtype=torch.long)

plt.figure(figsize=(4, 4))
colors = ['steelblue', 'coral', 'seagreen']
for cls_idx, name in enumerate(SHAPE_NAMES):
    mask = y == cls_idx
    plt.scatter(X[mask, 0], X[mask, 1], s=5, alpha=0.5, color=colors[cls_idx], label=name)
plt.legend()
plt.title('Three shape classes')
plt.axis('equal')
plt.show()

---

## Step 2: Forward Diffusion (recap from Part 1)

Same noise schedule and `q` function as always. Already working.

In [ ]:
T = 200
beta  = torch.linspace(1e-4, 0.02, T)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
sqrt_alpha_bar           = torch.sqrt(alpha_bar)
sqrt_one_minus_alpha_bar = torch.sqrt(1 - alpha_bar)
sqrt_alpha_inv           = torch.sqrt(1 / alpha)
pred_noise_coeff         = (1 - alpha) / sqrt_one_minus_alpha_bar

def q(x_0, t):
    noise = torch.randn_like(x_0)
    x_t = sqrt_alpha_bar[t][:, None] * x_0 + sqrt_one_minus_alpha_bar[t][:, None] * noise
    return x_t, noise

print('q defined.')

---

## Step 3: Reverse Diffusion (recap from Part 2)

Same reverse step formula as always, for 2D points. Already working.

In [ ]:
def reverse_step(x_t, t, e_t):
    sqrt_alpha_inv_t = sqrt_alpha_inv[t][:, None]
    coeff_t          = pred_noise_coeff[t][:, None]
    u_t = sqrt_alpha_inv_t * (x_t - coeff_t * e_t)
    if t[0] == 0:
        return u_t
    beta_prev = beta[t - 1][:, None]
    return u_t + torch.sqrt(beta_prev) * torch.randn_like(x_t)

print('reverse_step defined.')

---

## Step 4: Conditional Model and Training

The model now also takes a one-hot class vector, the same way Part 3's UNetCFG does. Training uses context dropout (10 percent of the time the class is hidden), same idea as Part 3. 10000 steps takes about 30-40 seconds on CPU. Already working.

In [ ]:
class CondMLP(nn.Module):
    def __init__(self, n_classes=N_CLASSES, hidden_dim=128, t_embed_dim=16, c_embed_dim=16):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, t_embed_dim), nn.GELU(),
            nn.Linear(t_embed_dim, t_embed_dim),
        )
        self.class_embed = nn.Sequential(
            nn.Linear(n_classes, c_embed_dim), nn.GELU(),
        )
        self.net = nn.Sequential(
            nn.Linear(2 + t_embed_dim + c_embed_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x, t_norm, c):
        t_emb = self.time_embed(t_norm.view(-1, 1))
        c_emb = self.class_embed(c)
        h = torch.cat([x, t_emb, c_emb], dim=1)
        return self.net(h)

model = CondMLP()
optimizer = Adam(model.parameters(), lr=1e-3)
STEPS = 10000
BATCH_SIZE = 256
DROP_PROB = 0.1

model.train()
for step in range(STEPS):
    idx = torch.randint(0, len(data), (BATCH_SIZE,))
    x_0 = data[idx]
    y_0 = labels[idx]
    t   = torch.randint(0, T, (BATCH_SIZE,))

    c_hot = nn.functional.one_hot(y_0, N_CLASSES).float()
    mask  = torch.bernoulli(torch.ones(BATCH_SIZE, 1) - DROP_PROB)
    c_in  = c_hot * mask

    x_noisy, noise = q(x_0, t)
    pred = model(x_noisy, t.float() / T, c_in)
    loss = nn.functional.mse_loss(pred, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (step + 1) % 2000 == 0:
        print(f'Step {step+1}/{STEPS} | loss: {loss.item():.4f}')

model.eval()
print('Training done.')

---

## Step 5: Conditional Sampling with CFG (your turn)

The model can produce both a conditional prediction (with the class) and an unconditional prediction (with a zeroed class vector). Combine them using the same classifier-free guidance formula from Part 3.

Right now the function ignores `w` completely (it just uses the conditional prediction). Fix the `# TODO` line.

In [ ]:
@torch.no_grad()
def sample_conditional(model, class_idx, w=1.0, n=300):
    """
    Generate n points conditioned on a class index, using classifier-free guidance.
    class_idx : which class to generate (0=circle, 1=spiral, 2=cross)
    w         : guidance weight (0.0 = unconditional, try 1.0 to 2.0 for stronger guidance)
    """
    c_hot  = torch.zeros(n, N_CLASSES); c_hot[:, class_idx] = 1.0
    c_zero = torch.zeros(n, N_CLASSES)

    x = torch.randn(n, 2)
    for i in reversed(range(T)):
        t_batch = torch.full((n,), i, dtype=torch.long)
        t_norm  = t_batch.float() / T

        e_cond   = model(x, t_norm, c_hot)
        e_uncond = model(x, t_norm, c_zero)

        # TODO: combine e_cond and e_uncond using guidance weight w
        # Same formula as Part 3: e_t = (1 + w) * e_cond - w * e_uncond
        e_t = e_cond   # placeholder, replace this line

        x = reverse_step(x, t_batch, e_t)
    return x

print('sample_conditional defined. Currently ignores w, fix the TODO above.')

<details>
<summary><b>Click to show the solution</b></summary>

```python
e_t = (1 + w) * e_cond - w * e_uncond
```
</details>

---

## Step 6: Generate and Visualize

This generates each shape class separately and shows them side by side. Already working, once Step 5 is fixed.

In [ ]:
fig, axes = plt.subplots(1, N_CLASSES, figsize=(12, 4))
for cls_idx, (ax, name) in enumerate(zip(axes, SHAPE_NAMES)):
    pts = sample_conditional(model, class_idx=cls_idx, w=1.5, n=300)
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5, color=colors[cls_idx])
    ax.set_title(f'{name} (w=1.5)')
    ax.axis('equal')
    ax.axis('off')
plt.suptitle('Conditionally generated shapes')
plt.tight_layout()
plt.show()

---

## Step 7: Try It Yourself

Design a 4th shape class of your own, add it to the `generators` list in Step 1, retrain, and regenerate. A starting idea for a star shape:

```python
# def make_star(n, noise=0.03):
#     k = 5   # number of points
#     angle = np.random.uniform(0, 2*np.pi, n)
#     r = 0.5 + 0.5 * np.cos(k * angle)
#     x = r * np.cos(angle) + np.random.randn(n) * noise
#     y = r * np.sin(angle) + np.random.randn(n) * noise
#     return np.stack([x, y], axis=1)
```

Remember to update `SHAPE_NAMES`, `N_CLASSES`, and `generators` together.

### Bonus (optional, if you have extra time)

Try negative guidance, the same idea as Part 3's bonus task: instead of pushing away from the unconditional prediction, push away from a different class. For example, generate a circle while pushing away from spiral:

```python
# e_circle  = model(x, t_norm, c_circle)
# e_spiral  = model(x, t_norm, c_spiral)
# e_t = (1 + w) * e_circle - w * e_spiral
```

Compare the result with standard CFG. Does it still look like a circle?

---

## Your Report

Answer these questions in this cell:

**1. Look at the Step 6 grid. At w=1.5, are the three shapes clearly distinguishable from each other?**

_(your answer here)_

**2. Try `w=0.0` for one class. What happens, and why does that match what `w=0.0` means in the formula?**

_(your answer here)_

**3. If you did Step 7 or the Bonus: what did you try, and what did you observe? If not: which one would you try first and why?**

_(your answer here)_